In [1]:
import sys, pandas as pd, json
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.load_cloudtrail import load_cloudtrail
from src.cloudtrail_adapter import adapt_cloudtrail_event


print("Project root:", project_root)

file_path = project_root / "data" / "synthetic_cloudtrail.json"

records = load_cloudtrail(str(file_path))

print("Records loaded: ", len(records))

Project root: /Users/gerardoramirez/Library/CloudStorage/GoogleDrive-gerardoram1010@gmail.com/My Drive/Portfolio Projects/cloud-security-intelligence-platform
Records loaded:  100


In [2]:
normalized_events = []

for record in records:
    normalized = adapt_cloudtrail_event(record)
    normalized_events.append(normalized)

Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.


In [3]:
df = pd.DataFrame(normalized_events)
df.head()

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record
0,2026-11-11T07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev..."
1,2026-11-10T02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev..."
2,2026-11-14T13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev..."
3,2026-11-12T08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev..."
4,2026-11-10T11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev..."


In [4]:
# Convert timestamps to UTC datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

In [5]:
# Using JSON seralization to create json strings of the record that can be checked for duplicates

df["raw_record_json"] = df["raw_record"].apply(
    lambda record: json.dumps(record, sort_keys=True)
)
df.head()

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record,raw_record_json
0,2026-11-11 07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""errorCode"": ""Acces..."
1,2026-11-10 02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""PutOb..."
2,2026-11-14 13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""PutOb..."
3,2026-11-12 08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""eventName"": ""Conso..."
4,2026-11-10 11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""Conso..."


In [6]:
df_clean = df.drop_duplicates(
    subset=["raw_record_json"]
)

In [7]:
print("Original records:", len(df))
print("Clean records:", len(df_clean))
print("Removed duplicates:", len(df) - len(df_clean))

Original records: 100
Clean records: 95
Removed duplicates: 5


In [8]:
df_clean = df_clean.copy()

df_clean["event_date"] = df_clean["timestamp"].dt.date

df_clean.head()

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record,raw_record_json,event_date
0,2026-11-11 07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""errorCode"": ""Acces...",2026-11-11
1,2026-11-10 02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""PutOb...",2026-11-10
2,2026-11-14 13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""PutOb...",2026-11-14
3,2026-11-12 08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""eventName"": ""Conso...",2026-11-12
4,2026-11-10 11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""Conso...",2026-11-10


In [9]:
event_counts = (
    df_clean.groupby(["event_date", "user"])
    .size()
    .reset_index(name="event_count")
)

event_counts.head(10)

,event_date,user,event_count
0,2026-11-10,admin,7
1,2026-11-10,analyst,2
2,2026-11-10,developer,5
3,2026-11-10,service-user,3
4,2026-11-11,admin,5
5,2026-11-11,analyst,5
6,2026-11-11,developer,4
7,2026-11-11,service-user,5
8,2026-11-12,admin,6
9,2026-11-12,analyst,3


In [10]:
print("Grouped event total:", event_counts["event_count"].sum())

Grouped event total: 95


In [16]:
# Calculating failure rate

df_clean["failed"] = df_clean["raw_record"].apply(
    lambda record: "errorCode" in record
)

df_clean

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record,raw_record_json,event_date,failed
0,2026-11-11 07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""errorCode"": ""Acces...",2026-11-11,True
1,2026-11-10 02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""PutOb...",2026-11-10,False
2,2026-11-14 13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""PutOb...",2026-11-14,False
3,2026-11-12 08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""eventName"": ""Conso...",2026-11-12,False
4,2026-11-10 11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev...","{""awsRegion"": ""us-east-1"", ""eventName"": ""Conso...",2026-11-10,False
...,...,...,...,...,...,...,...,...,...,...,...
90,2026-11-13 18:00:00+00:00,admin,GetObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-13T18:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-13T18:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""GetOb...",2026-11-13,False
91,2026-11-13 06:00:00+00:00,admin,GetObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-13T06:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-13T06:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""GetOb...",2026-11-13,False
92,2026-11-14 00:00:00+00:00,developer,PutObject,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T00:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-14T00:00:00+00:00', 'ev...","{""awsRegion"": ""eu-west-1"", ""eventName"": ""PutOb...",2026-11-14,False
93,2026-11-14 07:00:00+00:00,analyst,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-14T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-14T07:00:00+00:00', 'ev...","{""awsRegion"": ""us-west-2"", ""eventName"": ""PutOb...",2026-11-14,False


In [19]:
print(df_clean["failed"].value_counts())

print("Total failed events:", df_clean["failed"].sum())

failed
False    86
True      9
Name: count, dtype: int64
Total failed events: 9


In [20]:
failure_rate = df_clean["failed"].sum() / len(df_clean)

print("Failure Rate: ", failure_rate)

Failure Rate:  0.09473684210526316


In [27]:
daily_summary = (
    df_clean.groupby(["event_date", "user"])
    .agg(
        event_count=("failed", "size"),
        failure_count=("failed", "sum")
    )
    .reset_index()
)
daily_summary.head(5)

,event_date,user,event_count,failure_count
0,2026-11-10,admin,7,1
1,2026-11-10,analyst,2,0
2,2026-11-10,developer,5,0
3,2026-11-10,service-user,3,0
4,2026-11-11,admin,5,2


In [29]:
daily_summary["failure_rate"] = (
    
    daily_summary["failure_count"] /
    daily_summary["event_count"]
)

daily_summary.head(5)

,event_date,user,event_count,failure_count,failure_rate
0,2026-11-10,admin,7,1,0.142857
1,2026-11-10,analyst,2,0,0.000000
2,2026-11-10,developer,5,0,0.000000
3,2026-11-10,service-user,3,0,0.000000
4,2026-11-11,admin,5,2,0.400000
5,2026-11-11,analyst,5,0,0.000000
6,2026-11-11,developer,4,0,0.000000
7,2026-11-11,service-user,5,1,0.200000
8,2026-11-12,admin,6,0,0.000000
9,2026-11-12,analyst,3,0,0.000000


In [33]:
# Confirm that each failure rate is between 0 - 1
assert daily_summary["failure_rate"].between(0, 1).all()

In [38]:
# Aggregation validation. We should have 95 total count and 9 failed events

event_count_total = daily_summary["event_count"].sum()

failure_count_total = daily_summary["failure_count"].sum()

overall_failure_rate = failure_count_total / event_count_total

assert event_count_total == len(df_clean)

assert failure_count_total == df_clean["failed"].sum()

print("Event Count Total: ", event_count_total)
print("Failure Count Total: ", failure_count_total)
print(f"Overall Failure Rate: {overall_failure_rate:.2%}")

Event Count Total:  95
Failure Count Total:  9
Overall Failure Rate: 9.47%


### Day 14 — Security Event Analysis

Analyzed 100 synthetic CloudTrail events, removing 5 exact duplicates using JSON serialization and pandas. Converted timestamps to UTC calendar dates and classified failed events using the `errorCode` key. Grouped events by date and user to calculate activity counts, failure counts, and failure rates.

**Results:** 95 unique events, 9 failures, and a 9.47% overall failure rate. Validated aggregated totals and failure-rate bounds using assertions.

**Limitation:** These results represent synthetic data and do not reflect actual AWS security activity. Failed events do not necessarily indicate malicious behavior.